
# Rasters con valores bioclimáticos y precipitación anual

## bio *Bioclimáticos*
## prec *Precipitación*
# _________________________________________________________________________________________________________________

**Caragando librería e importando paquetes para uso en el código**


In [17]:
# Librerías y paquetes necesarios para poder operar los rasters.
!pip install geopandas rasterio pandas tqdm

import geopandas as gpd
import geopandas as gpf
import pandas as pd
import rasterio
import glob
import os
from tqdm import tqdm
import numpy as np
import pandas as pd


# *********************************************************************************************************************
# DIRECTORIOS 
## DEFINIENDO DIRECTORIOS
Se definen las rutas para la obtención de los datos (archivos .tiff) que contienen tod la información climática desde el año 1970 hasta el añ0 2000. 
# *********************************************************************************************************************

In [2]:
# Verifico si existe la carpeta
#ruta linux root_dir = "/home/guillermo/capas/"  Esta ruta es si ejecuto el código en servidor linux 
root_dir = "capas/"
if os.path.exists(root_dir) and os.path.isdir(root_dir): #Verificación si es una carpeta y no un archivo.
    print(f"La carpeta {root_dir} existe.")
    
    # Buscar todos los .tif en subcarpetas
    archivos = []
    for carpeta, subcarpetas, files in os.walk(root_dir):
        for f in files:
            if f.endswith(".tif"):
                archivos.append(os.path.join(carpeta, f))
    
    if archivos:
        print(f"Se encontraron {len(archivos)} archivos .tif en {root_dir} (incluyendo subcarpetas).")
    else:
        print(f"No se encontraron archivos .tif en {root_dir}.")
else:
    print(f"La carpeta {root_dir} no existe.")

La carpeta capas/ existe.
Se encontraron 232 archivos .tif en capas/ (incluyendo subcarpetas).


# *********************************************************************************************************************
## Listando los archivos.
Se crean una lista con los archivos **.tiff** y a su vez se crea la metadata del diccionario a partir de los acrónimos
# *********************************************************************************************************************

In [10]:
# Listando los archivos .tif en subcarpetas
raster_files = glob.glob(root_dir + "/**/*.tif", recursive=True)

# Creando los acrónimos automáticos (nombre del archivo sin extensión) para crear el diccionario
acronyms = [f.split("\\")[-1].replace(".tif", "") for f in raster_files]  # Windows path
# Si estoy Linux/Mac cambia a: f.split("/")[-1]


# *********************************************************************************************************************
## Obteniendo la información de las accesiones.
Se obtiene la información de las accesiones que han sido previamente extraida y transformada
# *********************************************************************************************************************

In [11]:
# Cargargando accesiones que ya fueron limpiadas (ejemplo CSV con lon y lat)
file_path = "./dataset_entrada/Dataset_Frijol.csv"

# Verificar si el archivo existe antes de cargarlo
if os.path.exists(file_path):
    try:
        accesiones = pd.read_csv(file_path)
        print(f"El archivo fue cargado correctamente: {file_path}")
        print(f"El dataset tiene {accesiones.shape[0]} filas y {accesiones.shape[1]} columnas.")
    except Exception as e:
        print(f"Error al cargar el archivo: {e}")
else:
    print(f"El archivo no existe en la ruta: {file_path}")

El archivo fue cargado correctamente: ./dataset_entrada/Dataset_Frijol.csv
El dataset tiene 3357 filas y 15 columnas.


# *********************************************************************************************************************
## Convirtiendo a GeoDataframe.
En este paso lo que hago es transformar el dataframe convencional de pandas (accesiones) en un GeoDataFrame de geopandas (gdf). Esto permite generar una estructura espacial para poder de manejar coordenadas geográficas. En otras palabras, lo que se hace aquí es transformar la tabla de datos tabulares en una capa geográfica donde cada registro se representa como un punto en el espacio (latitud y longitud).
Pandas maneja números y textos mientras Geopandas maneja geometrías. Esto me permite representar cada accesión como un punto geográfico dentro del sistema de referencia WGS84 (EPSG:4326), compatible con las capas raster de WorldClim y otras.
De esta forma, puedo realizar análisis espaciales, como la extracción de variables climáticas o la visualización de patrones ecológicos.

# *********************************************************************************************************************


In [12]:
# Convirtiendo las accesiones accesiones a GeoDataFrame

gdf = gpd.GeoDataFrame(
    accesiones,
    geometry=gpd.points_from_xy(accesiones.DECLONGITUDE, accesiones.DECLATITUDE), #Esta parte crea una columna de geometría usando las coordenadas x (longitud) y y (latitud).
    #crs significa Coordinate Reference System (Sistema de Referencia de Coordenadas). EPSG:4326 corresponde al sistema WGS84, usado por GPS y por la mayoría de capas globales (incluido WorldClim).
    crs="EPSG:4326" 
)

# *********************************************************************************************************************
## Extrayendo los valores de las capas rasters.
En este punto extraigo los valores de la capa rasters en los puntos de ubicación de las accesiones.
Por cada accesión (punto), busco en qué celda del raster cae y obtengo el valor de esa celda.
Esto constituye el enlace entre los datos genéticos/geográficos y los datos ambientales lo que permite construir una base enriquecida con variables climáticas que caracterizan el ambiente de origen de cada accesión.
En este paso o que hago es extarer del cada raster el valor asociado a cada punto de coordenada que se le crea a cada accesión  (gdf), esto me permite obtener todas las variables para cada accesión.
# *********************************************************************************************************************


In [13]:
# Extrayendo los valores de los rasters en puntos
def extraer_raster(rfile, gdf):
    #abre el archivo raster en modo lectura. El prefijo with asegura que el archivo se cierre automáticamente al finalizar.src es el objeto raster (una matriz de valores distribuidos en el espacio).
    with rasterio.open(rfile) as src:
        valores = [x[0] for x in src.sample(zip(gdf.geometry.x, gdf.geometry.y))]
    return valores

# *********************************************************************************************************************
## Extrayendo los valores de los rasters
Este bloque de código recorre automáticamente todas las capas raster para extraer sus valores climáticos en las coordenadas de las accesiones.
Luego, va almacenando los resultados en un DataFrame estructurado, donde cada columna representa una variable ambiental (por ejemplo, bio1, bio2, prec1, etc.), y cada fila corresponde a una accesión.
Aquí es donde se crea la matriz ambiental completa de todas las accesiones y variables.
# *********************************************************************************************************************


In [14]:
# Extrayendo los valores de todos los rasters 
valores = pd.DataFrame()

for i, rfile in enumerate(tqdm(raster_files, desc="Extrayendo rasters")):
    try:
        valores[acronyms[i]] = extraer_raster(rfile, gdf)
    except Exception as e:
        print(f"Error en {rfile}: {e}")
        valores[acronyms[i]] = [None] * len(gdf)


Extrayendo rasters:  43%|█████████████████████████▊                                  | 100/232 [01:42<03:36,  1.64s/it]C:\Users\guill\AppData\Local\Temp\ipykernel_9648\1106133152.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  valores[acronyms[i]] = extraer_raster(rfile, gdf)
Extrayendo rasters:  44%|██████████████████████████                                  | 101/232 [01:43<03:31,  1.61s/it]C:\Users\guill\AppData\Local\Temp\ipykernel_9648\1106133152.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  valores[acronyms[i]] =

In [15]:
valores.shape


(3357, 232)

# Uniendo la información
Una vez que he obtenido toda la información lo que hago es unir las dos fuentes en un Dataframe. Hago una integración de los datos (Accesiones <-> datos climáticos)
# *********************************************************************************************************************


In [18]:
# Limpiar nombres de columnas (quita rutas y extensiones para claridad en la metadata)
valores.columns = [str(col).split('/')[-1].split('\\')[-1].split('.')[0] for col in valores.columns]

# Detectar columnas numéricas
num_cols = valores.select_dtypes(include=[np.number]).columns

# Reemplazar valores inválidos solo en columnas numéricas
valores[num_cols] = valores[num_cols].replace([-3.4e+38, 3.4e+38], np.nan)

# Limpiar valores fuera de rango (solo numéricos)
valores[num_cols] = valores[num_cols].apply(
    lambda col: col.apply(lambda x: np.nan if isinstance(x, (int, float)) and (x < -1e5 or x > 1e5) else x)
)

print("Limpieza completada: nombres corregidos y valores extremos reemplazados.")

Limpieza completada: nombres corregidos y valores extremos reemplazados.


## Uniendo accesiones con variables ##
Lo que hago en este punto es crear la matriz ambiental, donde cada fila representa una accesión y cada columna representa una variable ambiental.
Esta esla base del enfoque FIGS: vincular diversidad genética ↔ ambiente de origen.
# *********************************************************************************************************************


In [20]:
# Uniendo las accesiones con los valores climáticos valores climáticos
resultado = pd.concat([accesiones, valores], axis=1)

## Guardando resultados ##
Una vez tengo asociada las accesiones con sus valores climáticos respectivos, procedo a guardar el nuevo .
# *********************************************************************************************************************


In [25]:
# Guardar el dataset limpio en un nuevo archivo Excel
# Este archivo es eld ataset para las siguientes fases (FIGS y ML)
output_filename_excel = './dataset_salida/accesiones_con_worldclim.xlsx'
resultado.to_excel(output_filename_excel, index=False)
output_filename_csv = './dataset_salida/accesiones_con_worldclim.csv'
resultado.to_csv(output_filename_csv, index=False)
print(f"Dataset limpio guardado ")

Dataset limpio guardado 


In [21]:
resultado.head()

,DOI,ACCENUMB,CURATION,GENUS,SPECIES,GRIN_NAME,SAMPSTAT,ACQDATE,ORIGCTY,COLLSITE,...,wind_12,wind_2,wind_3,wind_4,wind_5,wind_6,wind_7,wind_8,wind_9,wind_annual
0,10.18730/JKECH,G16570,FULL,Phaseolus,vulgaris,Phaseolus vulgaris,300,19780101,IRN,KARAJ - MARKAZI,...,2.0,2.5,3.2,3.5,3.0,2.8,2.9,2.7,2.5,255
1,10.18730/JKEKR,G16576,FULL,Phaseolus,vulgaris,Phaseolus vulgaris,300,19780101,IRN,KERMANSHAH CHAMCHAL - KERMANSHAH - KERMANSHAHAN,...,1.7,2.5,2.9,2.4,2.1,2.6,2.5,2.4,2.4,255
2,10.18730/JZXX0,G990,FULL,Phaseolus,vulgaris,Phaseolus vulgaris,300,19700901,IRN,AHWAZ - KHUZESTAN,...,1.8,3.0,2.8,2.9,2.8,3.7,3.5,3.1,2.6,255
3,10.18730/JZXWU,G989,FULL,Phaseolus,vulgaris,Phaseolus vulgaris,300,19700901,IRN,KAZERUN - FARS,...,1.8,2.5,3.1,3.4,3.5,3.2,3.2,3.0,2.5,255
4,10.18730/JZXT$,G987,FULL,Phaseolus,vulgaris,Phaseolus vulgaris,300,19700901,IRN,SHIRAZ - FARS,...,1.9,2.7,3.0,2.3,2.5,2.7,2.2,1.9,2.2,255
